# Camillion Claude RL Model — One-Click Train (Phase 1)
Run top to bottom. Reward is equity-only; alphas are suggestions the policy
learns to weight; diagnostics (Policy Doctor) are read-only.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## 2. Clone the repo


In [ ]:
!git clone https://github.com/monty313/Camillion_Claude_RL_model.git
%cd Camillion_Claude_RL_model


## 3. Install deps (TA-Lib optional — indicators also run pure-pandas)


In [ ]:
!pip -q install numpy pandas gymnasium stable-baselines3 torch


## 4. Run tests (verifies leak-free cache, reward discipline, 451 contract)


In [ ]:
!python tools/run_tests.py


## 5. Build the leakage-safe cache
Replace the synthetic demo with your MT5 1-minute OHLCV (index = bar open time).


In [ ]:
import numpy as np, pandas as pd
from src.data.cache_builder import build_cache, load_cache
n=20000; idx=pd.date_range('2024-01-01',periods=n,freq='1min')
close=100+np.cumsum(np.random.randn(n)*0.05)
df=pd.DataFrame({'open':close,'high':close+0.05,'low':close-0.05,'close':close,'volume':1.0},index=idx)
print(build_cache(df,'data_cache','EURUSD'))
ind,cl,t = load_cache('data_cache','EURUSD')


## 6. Register alphas (strategies -> alpha slots)


In [ ]:
from src.strategies.registry import AlphaRegistry
from src.strategies.examples import register_examples
def registry_factory():
    r=AlphaRegistry(); register_examples(r); return r


## 7. Train PPO (reward = equity change only)


In [ ]:
from src.training.trainer import train
model = train(np.asarray(ind), np.asarray(cl), np.asarray(t), registry_factory,
              total_timesteps=200_000, n_envs=8, window=5000, save_path='/content/drive/MyDrive/camillion_ppo')


## 8. Evaluate + Policy Doctor (read-only: what is the policy thinking?)


In [ ]:
from src.env.trading_env import TradingEnv
from src.training.trainer import sb3_policy_fn
from src.training.evaluate import evaluate_policy
from src.barbershop.policy_doctor import render
env = TradingEnv(np.asarray(ind),np.asarray(cl),np.asarray(t),registry_factory(),window=4000,random_window=False)
out = evaluate_policy(env, sb3_policy_fn(model), max_steps=3000, window=100)
print(render(out['policy_doctor']))


## 9. Save / Resume


In [ ]:
# from src.training.trainer import resume
# resume('/content/drive/MyDrive/camillion_ppo', np.asarray(ind),np.asarray(cl),np.asarray(t), registry_factory)
